# Boosted Decision Tree

In [ ]:
import pandas as pd
import numpy as np

from joblib import Parallel, delayed
import math
import tqdm
import tabulate
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.multioutput import MultiOutputClassifier

from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')
# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singola fold

In [ ]:
def fit_single_fold(train_idx, test_idx, features, target, groups, max_iter, max_depth, min_samples_leaf, learning_rate, l2_regularization, max_leaf_nodes, early_stopping):


    # ========== DEBUGGING: Stampo indici train/test  ==========

    """print("?"*50 + "\nDebug\n" + "?"*50)
    print(f"\nFold {fold} - File: {csv_name}")
    print(f"  Train indice: {train_index[:10]})")
    print(f"  Test indice: {test_index[:10]})")
    print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
    print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
    print("?"*100)"""
    # ==========================================================

    X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
    y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

    base_clf = HistGradientBoostingClassifier(
        max_iter=max_iter,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        learning_rate=learning_rate,
        l2_regularization=l2_regularization,
        max_leaf_nodes=max_leaf_nodes,
        early_stopping=early_stopping,
        random_state=42
    )

    rf = MultiOutputClassifier(base_clf, n_jobs=-1)

    # Addestro il modello sul train set di questa fold
    rf.fit(X_train, y_train)

    # Predico il target sul test set di questa fold
    y_pred = rf.predict(X_test)

    # DEBUG
    #score = f1_score(y_test, y_pred, average="micro")
    #print(f"Fold {fold} - {max_iter=}, {max_depth=}, {min_samples_leaf=}, Score={score:.4f}")

    return f1_score(y_test, y_pred, average="micro")

# Training


In [ ]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)
    
    # Mi definisco la lista dei target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']
    
    # Filtro solo le pazienti con PR valido
    df_validi = df.dropna(subset=original_target_list).copy()
    
    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')
    
    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione
    groups = df_validi['Patient ID']
    
    # Riempio a Nan se è rimasto vuoto
    features = features.fillna(features.mean())
    
    # Imposto la strategia di cross-validation
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Definisco gli iperparametri
    iperparametri = {
        'learning_rate': [0.05, 0.1],           # Tasso di apprendimento
        'max_iter': [100],                      # numero di iterazioni boosting
        'max_depth': [10, None],                # profondità massima degli alberi
        'min_samples_leaf': [5],                # min campioni in una foglia
        'l2_regularization': [0, 0.1],          # Penalizza pesi troppo grandi
        'max_leaf_nodes': [31],                 # Numero massimo di foglie per albero
        'early_stopping': ['auto']              # Arresto automatico
    }
    
    # Calcolo il numero totale di combinazioni
    total_combinations = math.prod(len(v) for v in iperparametri.values())
    
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")
    
    # Lista vuota per collezionare i punteggi di performance di ogni fold
    scores = []
    
    # Creo la barra di progresso con tqdm
    with tqdm.tqdm(total=total_combinations, 
                   desc="Combinazioni Testate",
                   bar_format="  {desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
                   leave=True, 
                   ncols=100) as pbar:
        
        # ciclo sui valori massimi delle iterazioni
        for learning_rate in iperparametri['learning_rate']:
            for max_iter in iperparametri['max_iter']:
                for max_depth in iperparametri['max_depth']:
                    for min_samples_leaf in iperparametri['min_samples_leaf']:
                        for l2_reg in iperparametri['l2_regularization']:
                            for max_leaf in iperparametri['max_leaf_nodes']:
                                for early_stop in iperparametri['early_stopping']:
                                    
                                    # Parallelizzo il training sulle fold
                                    fold_scores = Parallel(n_jobs=-1)(
                                        delayed(fit_single_fold)(train_idx, test_idx, features, target, groups, 
                                                                learning_rate=learning_rate, max_iter=max_iter,
                                                                max_depth=max_depth, min_samples_leaf=min_samples_leaf,
                                                                l2_regularization=l2_reg, max_leaf_nodes=max_leaf, 
                                                                early_stopping=early_stop) 
                                        for train_idx, test_idx in cv.split(features, target, groups)
                                    )
                                    
                                    # Calcolo media e deviazione standard degli score su tutte le fold
                                    mean_score = np.mean(fold_scores)
                                    std_score = np.std(fold_scores)
                                    
                                    # Aggiorno la barra con le metriche correnti
                                    pbar.set_postfix_str(
                                        f"F1: {mean_score:.3f} | LR: {learning_rate} | MD: {max_depth} | "
                                        f"L2: {l2_reg} | MaxIter: {max_iter}"
                                    )
                                    pbar.update(1)
                                    
                                    # Registro i risultati per la combinazione di parametri corrente
                                    scores.append({
                                        'learning_rate': learning_rate,
                                        'max_iter': max_iter,
                                        'max_depth': max_depth,
                                        'min_samples_leaf': min_samples_leaf,
                                        'l2_regularization': l2_reg,
                                        'max_leaf_nodes': max_leaf,
                                        'early_stopping': early_stop,
                                        'mean_score': mean_score,
                                        'std_score': std_score,
                                        'fold_scores': fold_scores
                                    })
    
    return scores

# Vado a stampare gli output in una maniera piú leggibile

In [ ]:
def print_results(results_per_dataset):
    '''
    Stampa i risultati della Grid Search in modo organizzato usando tabulate
    '''
    print("\n" + "=" * 80)
    print(" " * 20 + "RIEPILOGO DEI MIGLIORI RISULTATI")
    print("=" * 80)
    
    # Lista per il riepilogo finale comparativo
    summary_data = []
    
    for name, metrics_list in results_per_dataset.items():
        best_result = max(metrics_list, key=lambda x: x['mean_score'])
        
        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")
        
        # Tabella Iperparametri
        print("Iperparametri Ottimali:")
        params_table = [
            ['learning_rate', best_result['learning_rate']],
            ['max_iter', best_result['max_iter']],
            ['max_depth', best_result['max_depth']],
            ['min_samples_leaf', best_result['min_samples_leaf']],
            ['l2_regularization', best_result['l2_regularization']],
            ['max_leaf_nodes', best_result['max_leaf_nodes']],
            ['early_stopping', best_result['early_stopping']]
        ]
        print(tabulate.tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))
        print()
        
        # Aggiungi al riepilogo comparativo
        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            best_result['learning_rate'],
            best_result['max_depth'],
            best_result['l2_regularization'],
            best_result['max_iter']
        ])
    
    # Riepilogo Comparativo Finale
    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")
    
    # Ordina per F1-score decrescente
    summary_data.sort(key=lambda x: float(x[1]), reverse=True)
    
    print(tabulate.tabulate(summary_data, 
                   headers=['Dataset', 'F1-score', 'Std Dev', 'LR', 'Max Depth', 'L2 Reg', 'Max Iter'],
                   tablefmt='grid'))
    
    print("\n Analisi completata!\n")

# Lettura dei file

In [ ]:
# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati con tabulate
print_results(results_per_dataset)